In [ ]:
import pandas as pd
from helpers.pathlib import get_path_site_checkpoint

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow
from ata_pipeline1.preprocessors import SetRowIndex
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(7, THE_19TH.name))

In [ ]:
def get_session_distance(df):
    result = []
    events_with_leading = df.query(f"{FieldNew.NEWSLETTER_LEADING_EVENT}.isna() == False", engine="python")
    id_events_submission = events_with_leading.index
    id_events_leading = events_with_leading[FieldNew.NEWSLETTER_LEADING_EVENT]

    for id_event_submission, id_event_leading in zip(id_events_submission, id_events_leading):
        user_submit = df.at[id_event_submission, FieldSnowplow.DOMAIN_USERID]
        user_leading = df.at[id_event_leading, FieldSnowplow.DOMAIN_USERID]
        assert user_leading == user_submit

        session_submit = df.at[id_event_submission, FieldSnowplow.DOMAIN_SESSIONIDX]
        session_leading = df.at[id_event_leading, FieldSnowplow.DOMAIN_SESSIONIDX]

        result.append((user_submit, session_submit, session_leading, session_submit - session_leading))

    return pd.DataFrame(
        result, columns=[FieldSnowplow.DOMAIN_USERID, "session_submit", "session_leading", "session_diff"]
    ).set_index(FieldSnowplow.DOMAIN_USERID)

In [ ]:
df_result_afla = get_session_distance(df_afla)
df_result_dfp = get_session_distance(df_dfp)
df_result_ov = get_session_distance(df_ov)
df_result_19 = get_session_distance(df_19)

In [ ]:
assert (df_result_afla.session_diff >= 0).all()
assert (df_result_dfp.session_diff >= 0).all()
assert (df_result_ov.session_diff >= 0).all()
assert (df_result_19.session_diff >= 0).all()

In [ ]:
df_result_afla.session_diff.describe()

In [ ]:
df_result_dfp.session_diff.describe()

In [ ]:
df_result_ov.session_diff.describe()

In [ ]:
df_result_19.session_diff.describe()

In [ ]:
df_result_19.query("session_diff == 1")

In [ ]:
df_afla = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_afla)
df_dfp = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_dfp)
df_ov = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_ov)
df_19 = SetRowIndex(
    fields_index=[FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX, FieldNew.DOMAIN_SESSION_EVENTIDX],
    reset_index=True,
)(df_19)

In [ ]:
# There should be no leading events for this one
df_19.loc["87c422b6-8e78-4649-b9fc-83dad8dc36a3"]

In [ ]:
# There should be one leading event for this one, at session 1/visit 1
df_19.loc["9ec90287-2944-4842-849c-0dbeda995b00"]